# NY Times Letter Boxed Solver

**By Martino Gabra**  
An Automated solver for the NY Times Letter Boxed puzzle game

## Table of Contents
1. [Introduction](#introduction)
2. [Game Rules](#rules)
3. [Setup & Dependencies](#setup)
4. [Validation Functions](#validation)
5. [Dictionary Processing](#dictionary)
6. [Scoring Functions](#scoring)
7. [Solver Algorithm](#solver)
8. [Usage Example](#example)
9. [Try your own](#trial)

<a id='introduction'></a>
## Introduction

Letter Boxed is a word puzzle game in the New York Times where players have to link words from four distinct sides (the last letter of each word starts as the first letter of the next word). On each side there are 3 letters each and within 5 rounds, the user must have used every letter at least once. Here I provide a solution that finds the fewest words needed using a recursive algorithm.

This solver works by:
- Filtering the known English dictionary to only Letter boxed valid words for the current puzzle
- Selecting words that maximize unique letter coverage
- Recursively building word chains until all 12 letters are used then outputting the fewest word answers

---

## Notes

### Algorithm Performance
- For most puzzles, solutions are found in 2-4 words
- The recursion depth is limited by the requirement to use all 12 letters
- This does not find interesting words, only the quickest and shortest solution

### Limitations
- Assumes a solution can always be found
- Requires the pre-processed dictionary file to be available and it is hard-coded to be one specific file currently
- Assumes you have downloaded the correct reference file
- Does not handle puzzles with a recurring letter on more than one side
- Built only for the current iteration of Letter Boxed that operates on 12 letters but could be modified to be  
  n-length and n-sides

<a id='rules'></a>
## Game Rules

Letter Boxed has four key constraints:

1. **No consecutive letters from the same side** - Within one word, you cannot use two letters from the same side of the box consecutively. This also means no same letter duplicates.
2. **Word chaining** - The last letter of one word must be the first letter of the next word
3. **Use all letters** - You must use all 12 letters within five words
4. **Minimum word length** - Words must be at least 3 letters long

### Visual Example
```
        C   L   E
      ┌───────────┐
    P │           │ R
    V │           │ G
    T │           │ K
      └───────────┘
        O   A   U
        
```

Valid word: **PLUCK** (P→L→U→C→K alternates between sides)  
Invalid word: **GOAT** (O and A are both on the bottom side)

<a id='setup'></a>
## Setup & Dependencies

### Required Files
- `words_alpha.txt` - English dictionary from [dwyl/english-words](https://github.com/dwyl/english-words/blob/master/words_alpha.txt)
- `words_alphacurated_as_dict.txt` - Pre-processed dictionary (generated by this notebook)

### Data Source
This solver uses the English words dictionary available at:  
https://github.com/dwyl/english-words/blob/master/words_alpha.txt

<a id='validation'></a>
## Validation Functions

These functions validate whether words meet the game constraints.

In [68]:
def no_consecutive_duplicates(word):
    """
    Checks if this word has two consecutive letters that are identical. This function rule set is true
    irrespective of any valid grid or game so I filter the valid English dictionary using this function instead. 
    :param word: str
    :return: bool
    >>> no_consecutive_duplicates('spine')
    True
    >>> no_consecutive_duplicates('aardvark')
    False
    """
    n = 0
    while n+1 < len(word):
        if word[n] == word[n+1]:
            return False
        n += 1
    return True

In [69]:
def get_bins_for_letters(letters_this_game):
    """
    This is a helper function that gets the bin values for all of the letter sides:
    :param letters_this_game: a comma delimited string of 12 letters split into 3 letters each
    :return: dict
    >>> get_bins_for_letters('acf,drg,pot,qwe')
    {'1': 'acf', '2': 'drg', '3': 'pot', '4': 'qwe'}
    """
    final_dict = {}
    these_bins = letters_this_game.split(',')
    n = 1
    while n <= len(these_bins):
        final_dict[str(n)] = these_bins[n-1]
        n+=1
    return final_dict

In [70]:
def bin_choice_is_valid(dict_of_bins_this_game, this_word):
    """
    Checks if every consecutive letter in word is from a separate side of letters.

    :param dict_of_bins_this_game: dict of the_valid_letters_this_game
    :param this_word: a word to check
    :return: bool
    >>> bin_choice_is_valid({'1':'cdz', '2':'ask', '3':'frp', '4':'teg'}, 'cat')
    True
    >>> bin_choice_is_valid({'1':'cay','2':'ask', '3':'frp', '4':'zeg'}, 'cat')
    False
    """
    str_of_values = ''
    for one_letter in this_word:
        for one_bin in dict_of_bins_this_game:
            if one_letter in dict_of_bins_this_game[one_bin]:
                str_of_values += one_bin
    n = 0
    while n + 1 < len(str_of_values):
        if str_of_values[n] == str_of_values[n + 1]:
            return False
        n += 1
    return True

### Test the validation functions
#### Test the consecutive duplicates validation function

In [71]:
# Example usage
print("Is 'spine' a valid a valid letter boxed word? {}".format(no_consecutive_duplicates('spine')))
print("Is 'aardvark' a valid a valid letter boxed word? {}".format(no_consecutive_duplicates('aardvark')))

Is 'spine' a valid a valid letter boxed word? True
Is 'aardvark' a valid a valid letter boxed word? False


#### Test the bin choice validation function
##### Example: Check if 'cat' is valid for a given letter configuration of: "cdz,ask,frp,teg"


         (s1)C  D   Z  
          ┌───────────┐
    (s4)T │           │ A(s2)  
        E │           │ S  
        G │           │ K  
          └───────────┘
            F   R   P  
    


In [72]:
sample_bins = get_bins_for_letters('cdz,ask,frp,teg')
test_word = 'cat'
print(sample_bins)
print("Letter sides: {}".format(sample_bins))
print("Testing word: {}".format(test_word))
print("Is {} a valid word for this puzzle? {}". format(test_word, bin_choice_is_valid(sample_bins, test_word)))
print("Explanation: c(side 1) - a(side 2) - t(side 4)")

{'1': 'cdz', '2': 'ask', '3': 'frp', '4': 'teg'}
Letter sides: {'1': 'cdz', '2': 'ask', '3': 'frp', '4': 'teg'}
Testing word: cat
Is cat a valid word for this puzzle? True
Explanation: c(side 1) - a(side 2) - t(side 4)


<a id='dictionary'></a>
## Dictionary Processing

These functions handle the one-time setup of processing the raw word list into a usable dictionary, and loading it for each puzzle.

### One-Time Setup Function (Run Once)

This function processes the raw dictionary file and creates an optimized version. Only needs to be run once after downloading `words_alpha.txt`.

In [73]:
def convert_txt_to_dict():
    """
    One time script to organize glossary into a non-duplicate, >= 3 valid library.
    Takes in the format of: word1\nword2\nword3\n...wordn
    Returns the format of"letter1\tword1,word2,word3\nletter2\tword4,word5,wordn\n
    :return: None
    """
    final_dict = {}
    the_original_dict = './words_alpha.txt'
    with open(the_original_dict, 'r') as fi:
        for one_word in fi:
            one_word = one_word.strip()
            if len(one_word) >= 3 and no_consecutive_duplicates(one_word):
                if one_word[0] not in final_dict:
                    # this creates a dictionary of words, using the first letter as index and words as values
                    final_dict[one_word[0]] = [one_word]
                else:
                    final_dict[one_word[0]].append(one_word)

    with open(the_original_dict[:-4] + 'curated_as_dict.txt', 'w') as fo:
        for one_letter in final_dict:
            fo.write('{}\t{}\n'.format(one_letter, ",".join(final_dict[one_letter])))

In [74]:
# Ensure the path to words_alpha.txt is correctly referenced above and run the one-time dictionary setup:
convert_txt_to_dict()

### Load Pre-Processed Dictionary

In [75]:
def unpack_dictionary():
    """
    Unpacks the database from file and puts it into a dictionary of format: letter: [word1,word2,word3].
    :return: dict
    """

    original_dict_file = './words_alphacurated_as_dict.txt'
    original_dict = {}
    with open(original_dict_file, 'r') as fi:
        for line in fi:
            listed = line.strip().split('\t')
            original_dict[listed[0]] = listed[1].split(',')
    return original_dict

### Filter Dictionary for Current Puzzle

In [76]:
def modify_valid_dict(letters_this_game):
    """
    Takes 4 sets of letters and creates a total_dictionary of all valid words.
    1. No two letters from the same set can be consecutive.
    2. All letters in a word must exist in any of the side sets.
    :param letters_this_game: comma separated strings of letters
    :return: valid dict
    """

    all_words_dict = unpack_dictionary()
    word_bins = get_bins_for_letters(letters_this_game)
    total_letters_set = set(letters_this_game.replace(',', ''))
    final_dict = {}
    for first_letter in all_words_dict:
        if first_letter in total_letters_set:
            final_dict[first_letter] = []
            for one_word in all_words_dict[first_letter]:
                if all([this_letter in total_letters_set for this_letter in one_word]) and \
                        bin_choice_is_valid(word_bins, one_word):
                    final_dict[first_letter].append(one_word)
    return final_dict

<a id='scoring'></a>
## Scoring Functions

These functions calculate how many unique letters a word or word chain covers, which generates the optimal solution.

In [77]:
def unique_letter_count(this_string_word):
    """
    Counts number of unique letters for any valid or non-valid str of characters.
    :param this_string_word:
    :return:
    >>> unique_letter_count('aerates')
    5
    >>> unique_letter_count('cat')
    3
    """
    return len(set(this_string_word))

In [78]:
def max_word_score(list_of_words):
    """
    For a list of words, returns the max word(s) possible, with the score as a list of tuples.
    :param list_of_words:
    :return: list of tuples
    >>> max_word_score(['aerates', 'cat'])
    [('aerates', 5)]
    """
    final_tuples = []
    all_scores = [unique_letter_count(word.replace('.', '')) for word in list_of_words]
    max_score = max(all_scores)
    for one_position in enumerate(all_scores):
        if max_score == one_position[1]:
            final_tuples.append((list_of_words[one_position[0]], max_score))
    return final_tuples

### Scoring Example

In [79]:
sample_words = ['aerates', 'cat', 'computer']
print("Word scoring:")
for word in sample_words:
    score = unique_letter_count(word)
    print(f"  '{word}': {score} unique letters")

print(f"\nBest scoring word(s): {max_word_score(sample_words)}")

Word scoring:
  'aerates': 5 unique letters
  'cat': 3 unique letters
  'computer': 8 unique letters

Best scoring word(s): [('computer', 8)]


<a id='solver'></a>
## Solver Algorithm

The core solving logic uses a recursive approach:
1. Start with all valid words and pick those with maximum unique letters
2. For each high-scoring word, find all valid next words (starting with the last letter)
3. Calculate scores for the combined word chains
4. Keep only the highest-scoring chains and recurse
5. Stop when we find a chain covering all 12 letters


In [80]:
def add_words_to_valid_letter(this_games_valid_dict, this_word):

    full_dict = []
    for one_glossary_word in this_games_valid_dict[this_word[0][-1]]:
        full_dict.append(this_word[0] + '.' + one_glossary_word)
    return full_dict

In [81]:
def shortest_path(this_games_valid_dict, list_of_tuples):

    if not list_of_tuples:
        seeding_words = []
        for one_letter in this_games_valid_dict:
            seeding_words.extend(this_games_valid_dict[one_letter])
        list_of_tuples.extend(max_word_score(seeding_words))

    else:
        new_words = []
        for one_elem in list_of_tuples:
            new_words.extend(add_words_to_valid_letter(this_games_valid_dict, one_elem))
        list_of_tuples = max_word_score(new_words)
    words, scores = list(zip(*list_of_tuples))
    if 12 in scores:
        return list_of_tuples
    else:
        return shortest_path(this_games_valid_dict, list_of_tuples)

<a id='example'></a>
## Usage Example

Solving a valid Letter Boxed Puzzle.

```
        C   L   E
      ┌───────────┐
    P │           │ R
    V │           │ G
    T │           │ I
      └───────────┘
        O   A   U
        
```

### Puzzle Configuration

Enter the puzzle letters as four comma-separated groups (one for each side of the box):

In [82]:
puzzle_letters = 'cle,pvt,oau,rgi'

print("Solving puzzle with letters: {}".format(puzzle_letters))
print("Side 1: {}".format(puzzle_letters.split(',')[0].upper()))
print("Side 2: {}".format(puzzle_letters.split(',')[1].upper()))
print("Side 3: {}".format(puzzle_letters.split(',')[2].upper()))
print("Side 4: {}".format(puzzle_letters.split(',')[3].upper()))

Solving puzzle with letters: cle,pvt,oau,rgi
Side 1: CLE
Side 2: PVT
Side 3: OAU
Side 4: RGI


### Find Optimal Solution

In [83]:
# Generate valid word dictionary for this puzzle
this_games_dict = modify_valid_dict(puzzle_letters)

# Find best solution
solution = shortest_path(this_games_dict, [])

print("="*50)
print("Best Solutions Found")
print("="*50)
print("Number of solution(s): {}".format(len(solution)))
print("\n")
for word_chain, score in solution:
    words = word_chain.split('.')
    print(f"\n  {' - '.join(words)}")
    print(f"  ({len(words)} words, {score} unique letters)")

Best Solutions Found
Number of solution(s): 47



  overparticular - rag
  (2 words, 12 unique letters)

  overparticular - raga
  (2 words, 12 unique letters)

  overparticular - rage
  (2 words, 12 unique letters)

  overparticular - rager
  (2 words, 12 unique letters)

  overparticular - ragtag
  (2 words, 12 unique letters)

  overparticular - railage
  (2 words, 12 unique letters)

  overparticular - ravage
  (2 words, 12 unique letters)

  overparticular - ravager
  (2 words, 12 unique letters)

  overparticular - realgar
  (2 words, 12 unique letters)

  overparticular - reg
  (2 words, 12 unique letters)

  overparticular - regal
  (2 words, 12 unique letters)

  overparticular - regalia
  (2 words, 12 unique letters)

  overparticular - regalio
  (2 words, 12 unique letters)

  overparticular - regalo
  (2 words, 12 unique letters)

  overparticular - regave
  (2 words, 12 unique letters)

  overparticular - regear
  (2 words, 12 unique letters)

  overparticular - reget
  (2

<a id='trial'></a>
### Try your own!

Try another puzzle configuration:

In [85]:
# In your notebook, from 'Cell', Run All and make sure that there are no errors
# Put in the alternate_puzzle line in this block a list of lowercase letters as such: 'cle,pvt,oau,rgi'
# Uncomment the following to try a new puzzle:
# this_puzzle = ''
# this_dict = modify_valid_dict(this_puzzle)
# this_solution = shortest_path(this_dict, [])

# print("="*50)
# print("Best Solutions Found")
# print("="*50)
# print("Number of solution(s): {}".format(len(this_solution)))
# print("\n")
# for word_chain, score in this_solution:
#     words = word_chain.split('.')
#     print(f"\n  {' - '.join(words)}")
#     print(f"  ({len(words)} words, {score} unique letters)")